Loading packages to be used

In [1]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

Loading the features to train

In [2]:
# Reading the Dinov3 extracted features and classification csv files with matching cat ids
features = pd.read_csv("/home/oagile/hons_physics_uwc/hons_project_2026/dinov3_features_extracted_duplicated/all_dinov3_features.csv")  # Dinov3 features

In [3]:
rgz_classification = pd.read_csv("/home/oagile/hons_physics_uwc/hons_project_2026/data/DR1_FIRST_radio_classifications.csv")  # rgz classifications

# Reading the first five lines of each file
print("The first 5 lines of the Dinov3 features file:\n\n", features.head(), "\n\n")
print("The columns of the rgz classification file:\n\n", rgz_classification.columns, "\n\n")
print("The first 5 lines of the rgz classification file:\n\n", rgz_classification.head())




The first 5 lines of the Dinov3 features file:

    CatId                                           features
0      5  [ 3.79748166e-01  3.96371424e-01  4.94481891e-...
1      6  [ 6.39809445e-02  4.43955541e-01  6.40083849e-...
2      7  [-3.02132405e-02  3.91798913e-01  8.07184517e-...
3      8  [ 3.82982612e-01  2.38514155e-01  5.91870189e-...
4      9  [ 0.49267295  0.5601347   0.63945377 -1.079762... 


The columns of the rgz classification file:

 Index(['CatID', 'RGZID', 'ZooniverseID', 'RA', 'Dec', 'N_votes', 'N_total',
       'CL', 'N_comp', 'N_peaks', 'LAE', 'TSA', 'OL', 'TF', 'TF_err', 'DC'],
      dtype='str') 


The first 5 lines of the rgz classification file:

    CatID                 RGZID ZooniverseID         RA       Dec  N_votes  \
0      5  RGZ_J143121.3+232251   ARG0002565  217.83890  23.38094       38   
1      6  RGZ_J140600.1+232249   ARG0002566  211.50053  23.38030       39   
2      7  RGZ_J105512.0+232306   ARG000255x  163.79966  23.38497       57   
3      

Joining the two csv files in the same Cat Id

In [4]:
features = features.rename(columns={"CatId":"CatID"})  # Changing column to merge both files
merge_df = pd.merge(rgz_classification, features, on="CatID")
merge_df.columns


Index(['CatID', 'RGZID', 'ZooniverseID', 'RA', 'Dec', 'N_votes', 'N_total',
       'CL', 'N_comp', 'N_peaks', 'LAE', 'TSA', 'OL', 'TF', 'TF_err', 'DC',
       'features'],
      dtype='str')

Our targets for this model training is N_comp and N_peaks which represent the number of radio components and number of brightness peaks respectively. So we check for rare occurences in our targets before the train and test data splitting.

In [5]:
print(f"Checking for rare occurence in N_comp:\n\n {merge_df.loc[:, ['N_comp']].value_counts()}\n\n")
print(f"Checking for rare occurence in N_peaks:\n\n {merge_df.loc[:, ['N_peaks']].value_counts()}")

Checking for rare occurence in N_comp:

 N_comp
1         83038
2         14542
3          1814
4           142
5            25
6             3
7             2
Name: count, dtype: int64


Checking for rare occurence in N_peaks:

 N_peaks
1          64449
2          29603
3           4645
4            680
5            126
6             45
7             15
9              1
8              1
10             1
Name: count, dtype: int64


Now, we perform our train and test data splitting where we have sufficient occurences in our targets. Features extracted by Dinov3 are input variables while N_comps and N_peaks are our targets.

In [6]:
# Masking our dataframe where N_comp <=5 and N_peaks <=7.
cleaned_df = merge_df[(merge_df["N_comp"]<=5) & (merge_df["N_peaks"]<=7)]
cleaned_df.describe()


,CatID,RA,Dec,N_votes,N_total,CL,N_comp,N_peaks,LAE,TSA,OL,TF,TF_err
count,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000,99559.000000
mean,83068.978686,185.049242,28.971033,33.065650,37.210006,0.941635,1.187738,1.419158,26.721434,194.105899,0.809052,40.469845,0.216231
std,47493.139573,58.816894,17.622631,101.849708,117.930143,0.104245,0.447689,0.636870,17.403738,198.230145,0.154526,518.493384,0.148459
min,5.000000,0.029400,1.504820,1.000000,1.000000,0.650000,1.000000,1.000000,0.900000,1.890000,0.170000,0.110000,0.030000
25%,41957.500000,145.290545,12.927555,17.000000,17.000000,0.920000,1.000000,1.000000,15.800000,86.970000,0.750000,4.700000,0.150000
50%,82918.000000,185.015370,27.343060,23.000000,24.000000,1.000000,1.000000,1.000000,21.300000,141.800000,0.800000,12.180000,0.190000
75%,124212.500000,224.704415,43.927540,42.000000,47.000000,1.000000,1.000000,2.000000,31.200000,230.660000,0.850000,33.930000,0.250000
max,165416.000000,359.994700,64.667590,9412.000000,9569.000000,1.000000,5.000000,7.000000,184.200000,4930.750000,22.680000,113341.400000,21.900000


In [7]:
# Performing the train and test splitting
X, y = cleaned_df.loc[:, ["features"]], cleaned_df.loc[:, ["N_comp", "N_peaks"]]

The Dinov3 are of shape (99566, 1). So we turn them into the shape (99566, 768) for training.

In [8]:
feature_list = []

for i in range(len(X)):

    feature_list.append(X.iloc[i].str.strip('[]').str.split().apply(lambda x: [float(i) for i in x]))



In [11]:
feature_arr = np.array(feature_list)
feature_matrix = np.zeros((cleaned_df.shape[0], 768))


for i in range(cleaned_df.shape[0]):
    feature_matrix[i, :] = feature_arr[i, 0]

feature_matrix.shape

(99559, 768)

Spitting the training and testing data

In [14]:
X_train, X_test, y_train, y_test = train_test_split(feature_matrix, y, train_size=0.7)

In [15]:
y_train_comp, y_train_peaks = np.array(y_train.iloc[:, 0]), np.array(y_train.iloc[:, -1])  # output training data for N_comp and N_peaks respectively
y_test_comp, y_test_peaks = np.array(y_test.iloc[:, 0]), np.array(y_test.iloc[:, -1])  # output test data for N_comp and N_peaks respectively

Scaling the data to avoid the algorithms from underperforming

In [16]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)  # Applying the scaling to both the X_train and X_test data


Implementing the Multi-layer perceptron and Logistic  Regression

In [17]:
mlp_clf_comp, lr_clf_comp = MLPClassifier(), LogisticRegression()  # Multi-layer perceptron and logistic regression respectively for N_comp

mlp_clf_peaks, lr_clf_peaks = MLPClassifier(), LogisticRegression()  # Multi-layer perceptron and logistic regression respectively for N_peaks

Fitting the data in both the two models

In [18]:
# Training the model for N_comp
mlp_clf_comp.fit(X_train, y_train_comp)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",None


In [19]:
lr_clf_comp.fit(X_train, y_train_comp)

/home/oagile/anaconda3/envs/ml-uwc/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [20]:
# Training the model for N_peaks
mlp_clf_peaks.fit(X_train, y_train_peaks)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",None


In [21]:
lr_clf_peaks.fit(X_train, y_train_peaks)

/home/oagile/anaconda3/envs/ml-uwc/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [22]:
from sklearn.metrics import accuracy_score

Making predictions for each model

In [23]:
# Predicting the model for N_comp
y_pred_comp_mlp = mlp_clf_comp.predict(X_test)
print(accuracy_score(y_test_comp, y_pred_comp_mlp))

0.9032744073925271


In [24]:
y_pred_comp_lr = lr_clf_comp.predict(X_test)
print(accuracy_score(y_test_comp, y_pred_comp_lr))

0.9052162849872774


In [25]:
# Predicting the model for N_peaks
y_pred_peaks_mlp = mlp_clf_peaks.predict(X_test)
print(accuracy_score(y_test_peaks, y_pred_peaks_mlp))

0.8115374313646713


In [26]:
y_pred_peaks_lr = lr_clf_peaks.predict(X_test)
print(accuracy_score(y_test_peaks, y_pred_peaks_lr))

0.8210459354493103


Computing the auc 

In [27]:
# Probabilities for N_comp for both models
probs_comp_mlp  = mlp_clf_comp.predict_proba(X_test)
probs_comp_lr = lr_clf_comp.predict_proba(X_test)

# Probabilities for N_peaks for both models
probs_peaks_mlp  = mlp_clf_peaks.predict_proba(X_test)
probs_peaks_lr = lr_clf_peaks.predict_proba(X_test)

In [29]:
# auc for N_comp for both models
auc_comp_mlp  = roc_auc_score(y_test_comp, probs_comp_mlp, multi_class="ovo")
print('Area under curve for N_comp for Multi-layer perceptron is:', auc_comp_mlp)

auc_comp_lr = roc_auc_score(y_test_comp, probs_comp_lr, multi_class="ovo")
print('Area under curve for N_comp for Logistic regression is:', auc_comp_lr)

# auc for N_peaks for both models
auc_peaks_mlp  = roc_auc_score(y_test_peaks, probs_peaks_mlp, multi_class="ovo")
print('Area under curve for N_peaks for Multi-layer perceptron', auc_peaks_mlp)

auc_peaks_lr = roc_auc_score(y_test_peaks, probs_peaks_lr, multi_class="ovo")
print('Area under curve for N_peaks for Logistic regression', auc_peaks_lr)


Area under curve for N_comp for Multi-layer perceptron is: 0.7637436706813105
Area under curve for N_comp for Logistic regression is: 0.7305588546598594
Area under curve for N_peaks for Multi-layer perceptron 0.7971896467082247
Area under curve for N_peaks for Logistic regression 0.7726407781623541


Computing precision, recall and F1 score

In [34]:
# precision for N_comp for both models
precision_comp_mlp = precision_score(y_test_comp, y_pred_comp_mlp, average="weighted", zero_division=0)
print('The precision score for N_comp for Multi-layer perceptron is:', precision_comp_mlp)

precision_comp_lr = precision_score(y_test_comp, y_pred_comp_lr, average="weighted", zero_division=0)
print('The precision score for N_comp for Logistic regression is:', precision_comp_lr)

# precision for N_peaks for both models
precision_peaks_mlp  = precision_score(y_test_peaks, y_pred_peaks_mlp, average="weighted", zero_division=0)
print('The precision score for N_peaks for Multi-layer perceptron', precision_peaks_mlp)

precision_peaks_lr = precision_score(y_test_peaks, y_pred_peaks_lr, average="weighted", zero_division=0)
print('The precision score for N_peaks for Logistic regression', precision_peaks_lr)

The precision score for N_comp for Multi-layer perceptron is: 0.9027899724786266
The precision score for N_comp for Logistic regression is: 0.8947785088571191
The precision score for N_peaks for Multi-layer perceptron 0.8097697921485352
The precision score for N_peaks for Logistic regression 0.8096296692614984


In [35]:
# recall for N_comp for both models
recall_comp_mlp = recall_score(y_test_comp, y_pred_comp_mlp, average="weighted", zero_division=0)
print('The recall score for N_comp for Multi-layer perceptron is:', recall_comp_mlp)

recall_comp_lr = recall_score(y_test_comp, y_pred_comp_lr, average="weighted", zero_division=0)
print('The recall score for N_comp for Logistic regression is:', recall_comp_lr)

# recall for N_peaks for both models
recall_peaks_mlp  = recall_score(y_test_peaks, y_pred_peaks_mlp, average="weighted", zero_division=0)
print('The recall score for N_peaks for Multi-layer perceptron', recall_peaks_mlp)

recall_peaks_lr = recall_score(y_test_peaks, y_pred_peaks_lr, average="weighted", zero_division=0)
print('The recall score for N_peaks for Logistic regression', recall_peaks_lr)

The recall score for N_comp for Multi-layer perceptron is: 0.9032744073925271
The recall score for N_comp for Logistic regression is: 0.9052162849872774
The recall score for N_peaks for Multi-layer perceptron 0.8115374313646713
The recall score for N_peaks for Logistic regression 0.8210459354493103


In [36]:
# f1 score for N_comp for both models
f1_comp_mlp = f1_score(y_test_comp, y_pred_comp_mlp, average="weighted", zero_division=0)
print('The f1 score for N_comp for Multi-layer perceptron is:', f1_comp_mlp)

f1_comp_lr = f1_score(y_test_comp, y_pred_comp_lr, average="weighted", zero_division=0)
print('The f1 score for N_comp for Logistic regression is:', f1_comp_lr)

# f1 for N_peaks for both models
f1_peaks_mlp  = f1_score(y_test_peaks, y_pred_peaks_mlp, average="weighted", zero_division=0)
print('The f1 score for N_peaks for Multi-layer perceptron', f1_peaks_mlp)

f1_peaks_lr = f1_score(y_test_peaks, y_pred_peaks_lr, average="weighted", zero_division=0)
print('The f1 score for N_peaks for Logistic regression', f1_peaks_lr)

The f1 score for N_comp for Multi-layer perceptron is: 0.9030099220451742
The f1 score for N_comp for Logistic regression is: 0.8980651011173332
The f1 score for N_peaks for Multi-layer perceptron 0.8103799350861065
The f1 score for N_peaks for Logistic regression 0.8130777734439246


Optimising hyperparameters